In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

from docling.document_converter import DocumentConverter
from pathlib import Path


In [ ]:
@dataclass
class Section:
    heading: str
    section_number: Optional[str]
    level: int
    page_start: Optional[int]
    page_end: Optional[int]
    text: str


@dataclass
class Paper:
    paper_id: str
    title: str
    authors: List[str]
    year: int
    source_file: str
    sections: List[Section] = field(default_factory=list)

In [ ]:
PROJECT_ROOT = Path.cwd().parent
PAPER_DIR = PROJECT_ROOT / "data" / "papers"

pdf_path = PAPER_DIR / "01_rag_lewis_2020.pdf"

converter = DocumentConverter()
result = converter.convert(pdf_path)
doc = result.document

In [ ]:
for item, level in doc.iterate_items():
    item_type = type(item).__name__
    text = getattr(item, 'text', '')

    if 'SectionHeader' in item_type:
        print(level, item_type, text)

In [ ]:
import re

def get_section_level(heading: str) -> int:
    match = re.match(r"^(\d+(?:\.\d+)*)\s+", heading)

    if not match:
        return 1

    section_number = match.group(1)

    return section_number.count(".") + 1

def get_section_number(heading: str):
    match = re.match(r"^(\d+(?:\.\d+)*)\s+", heading)

    if match:
        return match.group(1)

    return None

In [ ]:
test_headings = [
    "1 Introduction",
    "2.1 Models",
    "4.5 Additional Results",
    "Abstract",
    "References"
]

for heading in test_headings:
    print(
        heading,
        "-> number:",
        get_section_number(heading),
        "| level:",
        get_section_level(heading)
    )

In [ ]:
sections = []

current_section = None

for item, level in doc.iterate_items():
    item_type = type(item).__name__
    text = getattr(item, "text", "").strip()

    if not text:
        continue

    if "SectionHeader" in item_type:
        current_section = {
            "heading": text,
            "section_number": get_section_number(text),
            "level": get_section_level(text),
            "text_parts": []
        }

        sections.append(current_section)

    elif current_section is not None:
        current_section["text_parts"].append(text)

In [ ]:
for section in sections:
    section["text"] = "\n\n".join(section["text_parts"])

In [ ]:
for section in sections[:5]:
    print("HEADING:", section["heading"])
    print("NUMBER:", section["section_number"])
    print("LEVEL:", section["level"])
    print("TEXT:", section["text"][:500])
    print("-" * 80)

In [ ]:
for item, level in doc.iterate_items():
    text = getattr(item, "text", "").strip()

    if text:
        print("TEXT:", text[:100])
        print("PROV:", getattr(item, "prov", None))
        break

In [ ]:
sections = []

current_section = None

for item, level in doc.iterate_items():
    item_type = type(item).__name__
    text = getattr(item, "text", "").strip()

    if not text:
        continue

    prov = getattr(item, "prov", [])
    page_no = prov[0].page_no if prov else None

    if "SectionHeader" in item_type:
        current_section = {
            "heading": text,
            "section_number": get_section_number(text),
            "level": get_section_level(text),
            "pages": [],
            "text_parts": []
        }

        if page_no is not None:
            current_section["pages"].append(page_no)

        sections.append(current_section)

    elif current_section is not None:
        current_section["text_parts"].append(text)

        if page_no is not None:
            current_section["pages"].append(page_no)

In [ ]:
for section in sections:
    section["text"] = "\n\n".join(section["text_parts"])

    if section["pages"]:
        section["page_start"] = min(section["pages"])
        section["page_end"] = max(section["pages"])
    else:
        section["page_start"] = None
        section["page_end"] = None

In [ ]:
for section in sections[:5]:
    print(
        section["heading"],
        "->",
        section["page_start"],
        "-",
        section["page_end"]
    )

In [ ]:
section_objects = []

for section in sections:
    section_obj = Section(
        heading=section["heading"],
        section_number=section["section_number"],
        level=section["level"],
        page_start=section["page_start"],
        page_end=section["page_end"],
        text=section["text"]
    )

    section_objects.append(section_obj)

In [ ]:
print(section_objects[1].heading)
print(section_objects[1].page_start)
print(section_objects[1].text[:300])

In [ ]:
paper = Paper(
    paper_id="rag_lewis_2020",
    title="Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
    authors=[],
    year=2020,
    source_file="01_rag_lewis_2020.pdf",
    sections=section_objects
)

In [ ]:
print(paper.title)
print(paper.year)
print(len(paper.sections))
print(paper.sections[0].heading)
print(paper.sections[0].page_start)

In [ ]:
def build_sections(doc):
    sections = []
    current_section = None

    for item, level in doc.iterate_items():
        item_type = type(item).__name__
        text = getattr(item, "text", "").strip()

        if not text:
            continue

        prov = getattr(item, "prov", [])
        page_no = prov[0].page_no if prov else None

        if "SectionHeader" in item_type:
            current_section = {
                "heading": text,
                "section_number": get_section_number(text),
                "level": get_section_level(text),
                "pages": [],
                "text_parts": []
            }

            if page_no is not None:
                current_section["pages"].append(page_no)

            sections.append(current_section)

        elif current_section is not None:
            current_section["text_parts"].append(text)

            if page_no is not None:
                current_section["pages"].append(page_no)

    section_objects = []

    for section in sections:
        text = "\n\n".join(section["text_parts"])

        page_start = min(section["pages"]) if section["pages"] else None
        page_end = max(section["pages"]) if section["pages"] else None

        section_objects.append(
            Section(
                heading=section["heading"],
                section_number=section["section_number"],
                level=section["level"],
                page_start=page_start,
                page_end=page_end,
                text=text
            )
        )

    return section_objects

In [ ]:
sections_test = build_sections(doc)

print(len(sections_test))
print(sections_test[0])
print(sections_test[1])

In [ ]:
def build_paper(
    doc,
    paper_id: str,
    title: str,
    authors: list[str],
    year: int,
    source_file: str
):
    sections = build_sections(doc)

    return Paper(
        paper_id=paper_id,
        title=title,
        authors=authors,
        year=year,
        source_file=source_file,
        sections=sections
    )

In [ ]:
paper_metadata = {
    "01_rag_lewis_2020.pdf": {
        "paper_id": "rag_lewis_2020",
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "authors": [],
        "year": 2020,
    },

    "02_dpr_karpukhin_2020.pdf": {
        "paper_id": "dpr_karpukhin_2020",
        "title": "Dense Passage Retrieval for Open-Domain Question Answering",
        "authors": [],
        "year": 2020,
    },

    "03_colbert_khattab_2020.pdf": {
        "paper_id": "colbert_khattab_2020",
        "title": "Efficient and Effective Passage Search via Contextualized Late Interaction over BERT",
        "authors": [],
        "year": 2020,
    },    

    "04_hyde_gao_2023.pdf": {
        "paper_id": "hyde_gao_2023",
        "title": "Precise Zero-Shot Dense Retrieval without Relevance Labels",
        "authors": [],
        "year": 2023,
    }, 

     "05_rag_survey_gao_2023.pdf": {
        "paper_id": "rag_survey_gao_2023",
        "title": "Retrieval-Augmented Generation for Large Language Models: A Survey",
        "authors": [],
        "year": 2023,
    },

    "06_self_rag_asai_2023.pdf": {
        "paper_id": "self_rag_asai_2023",
        "title": "SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE THROUGH SELF-REFLECTION",
        "authors": [],
        "year": 2023,
    },

    "07_lost_in_middle_liu_2023.pdf": {
        "paper_id": "lost_in_middle_liu_2023",
        "title": "Lost in the Middle: How Language Models Use Long Contexts",
        "authors": [],
        "year": 2023,
    },    


    "08_raptor_sarthi_2024.pdf": {
        "paper_id": "raptor_sarthi_2024",
        "title": "RAPTOR: RECURSIVE ABSTRACTIVE PROCESSING FOR TREE-ORGANIZED RETRIEVAL",
        "authors": [],
        "year": 2024,
    },    


    "09_crag_yan_2024.pdf": {
        "paper_id": "crag_yan_2024",
        "title": "Corrective Retrieval Augmented Generation",
        "authors": [],
        "year": 2024,
    },    

    "10_qwen3_embedding_2025.pdf": {
        "paper_id": "qwen3_embedding_2025",
        "title": "Qwen3 Embedding: Advancing Text Embedding and Reranking Through Foundation Models",
        "authors": [],
        "year": 2025,
    }
}

In [ ]:
papers = []

for file in PAPER_DIR.glob("*.pdf"):
    metadata = paper_metadata[file.name]

    result = converter.convert(file)
    doc = result.document

    paper = build_paper(
        doc=doc,
        paper_id=metadata["paper_id"],
        title=metadata["title"],
        authors=metadata["authors"],
        year=metadata["year"],
        source_file=file.name
    )

    papers.append(paper)

    print(
        f"Built: {paper.paper_id} | "
        f"sections: {len(paper.sections)}"
    )

In [ ]:
print("Total papers:", len(papers))

In [ ]:
print(papers[0].title)
print(len(papers[0].sections))
print(papers[0].sections[0].heading)